In [1]:
import os
import gc
import random
import warnings
from tqdm import tqdm
from PIL import Image

import numpy as np 
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet50, ResNet50_Weights

In [2]:
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
    
    print(f"Random seed set to {seed}")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except:
        pass

In [3]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [4]:
CRC100K_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/NCT-CRC-HE-100K/NCT-CRC-HE-100K"
PATHMNIST_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

BATCH_SIZE = 256
IMAGE_SIZE = (224, 224)

BUDGET = 200
MAX_ITERATIONS = 5

In [5]:
class NPZDataset(Dataset):
    def __init__(self, npz_path, split="train", transform=None):
        data = np.load(npz_path)
        
        if split == "train":
            self.img = np.concatenate((data["train_images"], data["val_images"]), axis=0)
            self.lbl = np.concatenate((data["train_labels"], data["val_labels"]), axis=0).squeeze()
        elif split == "test":
            self.img = data["test_images"]
            self.lbl = data["test_labels"].squeeze()
        else:
            raise ValueError("Split must be 'train' or 'test'")
            
        self.transform = transform
        
        self.classes = [
            "adipose", "background", "debris", "lymphocytes", "mucus", 
            "smooth_muscle", "normal_colon_mucosa", "cancer_associated_stroma", 
            "colorectal_adenocarcinoma"
        ]

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        img = self.img[idx]
        label = self.lbl[idx]
            
        img = Image.fromarray(img)
            
        if self.transform:
            img = self.transform(img)
            
        return img, label

In [6]:
def get_data_loaders(data_path, 
                     batch_size=BATCH_SIZE, 
                     image_size=IMAGE_SIZE,
                     seed=RANDOM_SEED):
    transform = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    num_cores = min(4, os.cpu_count() or 4) 
    
    if data_path.endswith(".npz"):
        train_dataset = NPZDataset(npz_path=data_path, split="train", transform=transform)
        test_dataset = NPZDataset(npz_path=data_path, split="test", transform=transform)
        class_names = train_dataset.classes
    else:
        full_dataset = ImageFolder(root=data_path, transform=transform)
        class_names = full_dataset.classes
        
        total_size = len(full_dataset)
        train_size = int(0.8 * total_size)
        test_size = total_size - train_size
        
        generator = torch.Generator().manual_seed(seed)
        train_dataset, test_dataset = random_split(
            full_dataset, 
            [train_size, test_size], 
            generator=generator
        )

    print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_cores,     
        pin_memory=True,        
        prefetch_factor=2,         
        persistent_workers=False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_cores,     
        pin_memory=True,        
        prefetch_factor=2,         
        persistent_workers=False
    )
    
    return train_loader, test_loader, class_names

In [7]:
from transformers import CLIPVisionModelWithProjection
def extract_embeddings(dataloader, device=DEVICE):
    print("Loading PLIP model...")
    model_name = "vinid/plip" 
    
    model = CLIPVisionModelWithProjection.from_pretrained(model_name).to(device)
    model.eval()

    all_embeddings = []
    true_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Extracting Embeddings"):
            images = images.to(device)
            
            outputs = model(pixel_values=images)
            
            features = outputs.image_embeds
            
            features = features / features.norm(p=2, dim=-1, keepdim=True)
            
            all_embeddings.append(features.cpu().numpy())
            true_labels.append(labels.numpy())

    embeddings = np.vstack(all_embeddings)
    true_labels = np.concatenate(true_labels)
    
    del model
    clear_memory()
    
    return embeddings, true_labels

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.cluster import KMeans
from tqdm import tqdm

class ALFAMixSelector:
    def __init__(self, budget, device="cuda"):
        self.budget = budget
        self.device = device

    def get_anchors(self, labeled_features, labeled_labels, num_classes):
        anchors = []
        for i in range(num_classes):
            mask = (labeled_labels == i)
            if mask.any():
                anchors.append(labeled_features[mask].mean(dim=0))
            else:
                anchors.append(labeled_features.mean(dim=0))
        return torch.stack(anchors)

    def select(self, unlabeled_features, labeled_features, labeled_labels, num_classes):

        unlabeled_features = unlabeled_features.to(self.device).requires_grad_(True)
        num_samples = unlabeled_features.shape[0]
        dim = unlabeled_features.shape[1]
        
        epsilon = 0.2 / np.sqrt(dim)
        
        proxy_classifier = nn.Linear(dim, num_classes).to(self.device)
        optimizer = torch.optim.Adam(proxy_classifier.parameters(), lr=0.01)
        
        for _ in range(10):
            logits = proxy_classifier(labeled_features.to(self.device))
            loss = F.cross_entropy(logits, labeled_labels.to(self.device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        proxy_classifier.eval()
        
        anchors = self.get_anchors(labeled_features, labeled_labels, num_classes).to(self.device)
        
        candidate_indices = []
        
        outputs = proxy_classifier(unlabeled_features)
        pseudo_labels = outputs.argmax(dim=1)
        
        loss = F.cross_entropy(outputs, pseudo_labels)
        loss.backward()
        grads = unlabeled_features.grad.data # (N, D)
        
        with torch.no_grad():
            for i in range(num_samples):
                z_u = unlabeled_features[i]
                g_u = grads[i]
                y_u = pseudo_labels[i]
                
                found_inconsistency = False
                for j in range(num_classes):
                    z_star = anchors[j]
                    diff = z_star - z_u
                    
                    norm_grad = torch.norm(g_u) + 1e-8
                    norm_diff = torch.norm(diff) + 1e-8
                    
                    alpha = epsilon * (norm_diff / norm_grad) * (g_u / (diff + 1e-8))
                    alpha = torch.clamp(alpha, 0, 0.5) # Giới hạn tỷ lệ trộn
                    
                    z_tilde = alpha * z_star + (1 - alpha) * z_u
                    
                    # Kiểm tra xem dự đoán có bị thay đổi (flip) không
                    y_tilde = proxy_classifier(z_tilde.unsqueeze(0)).argmax(dim=1)
                    if y_tilde != y_u:
                        found_inconsistency = True
                        break
                
                if found_inconsistency:
                    candidate_indices.append(i)

        # 5. Đa dạng hóa bằng Clustering (K-Means) [cite: 3216, 3218]
        if len(candidate_indices) <= self.budget:
            return np.array(candidate_indices)
        
        candidate_features = unlabeled_features[candidate_indices].detach().cpu().numpy()
        kmeans = KMeans(n_clusters=self.budget, random_state=42, n_init="auto")
        kmeans.fit(candidate_features)
        
        # Chọn mẫu gần tâm cụm nhất để gán nhãn [cite: 3133, 3668]
        from sklearn.metrics import pairwise_distances_argmin_min
        closest_indices, _ = pairwise_distances_argmin_min(kmeans.cluster_centers_, candidate_features)
        
        final_indices = [candidate_indices[idx] for idx in closest_indices]
        return np.array(final_indices)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
class ActiveFTSelector:
    def __init__(self, budget, lambda_reg=1.0, temperature=0.07, iterations=100, lr=1e-3):
        self.budget = budget
        self.lambda_reg = lambda_reg
        self.tau = temperature
        self.iterations = iterations
        self.lr = lr
    def select(self, features):
        """
        Choose the index of the appropriate samples"""
        device = features.device
        num_samples = features.shape[0]
        # Khởi tạo B mẫu để làm tham số theta
        indices = torch.randperm(num_samples)[:self.budget]
        theta = features[indices].detach().clone().requires_grad_(True)
        optimizer = torch.optim.Adam([theta], lr=self.lr)
        for _ in tqdm(range(self.iterations)):
            # Chuẩn hóa theta để nằm trên mặt cầu đơn vị
            with torch.no_grad():
                theta.data = F.normalize(theta.data, p=2, dim=1)
            sim_matrix = torch.matmul(features, theta.t()) / self.tau

            #Tìm theta gần nhất cho mỗi mẫu f_i
            max_sim, _ = torch.max(sim_matrix, dim=1)
            loss_dist = -torch.mean(max_sim)

            # Tính độ tương đồng giữa các theta với nhau
            theta_sim = torch.matmul(theta, theta.t()) / self.tau
            mask = ~torch.eye(self.budget, device=device).bool()
            theta_sim_filtered = theta_sim[mask].view(self.budget, -1)

            loss_reg = torch.mean(torch.log(torch.sum(torch.exp(theta_sim_filtered), dim=1)))
            total_loss = loss_dist + self.lambda_reg * loss_reg
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
        # Sau khi tối ưu, lựa chọn mẫu thực tế gần nhất với theta cuối cùng
        with torch.no_grad():
            theta_final = F.normalize(theta, p=2, dim=1)
            final_sim = torch.matmul(features, theta_final.t())
            selected_indices = torch.argmax(final_sim, dim=0)
        return selected_indices.cpu().numpy()
        

In [10]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

class TypiClustSelector:
    def __init__(self, k_nn=20):
        """
        Khởi tạo TypiClust.
        k_nn: Số lượng láng giềng để tính toán độ điển hình (Typicality), bài báo thường dùng 20.
        """
        self.k_nn = k_nn

    def get_typicality(self, features):
        """
        Tính toán độ điển hình (Typicality) cho toàn bộ mẫu.
        Typicality tỷ lệ nghịch với khoảng cách trung bình đến K láng giềng gần nhất.
        """
        # Sử dụng thuật toán k-NN để tìm khoảng cách
        nbrs = NearestNeighbors(n_neighbors=self.k_nn + 1, algorithm='auto').fit(features)
        distances, _ = nbrs.kneighbors(features)
        
        # Bỏ qua chính nó (cột 0), tính khoảng cách trung bình tới K láng giềng
        avg_distances = distances[:, 1:].mean(axis=1)
        
        # Typicality = 1 / (khoảng cách trung bình). Khoảng cách càng nhỏ -> Càng điển hình.
        typicality = 1.0 / (avg_distances + 1e-8)
        return typicality

    def select(self, all_features, is_labeled, batch_size):
        """
        Chọn mẫu theo chiến lược TypiClust.
        
        Args:
            all_features: Đặc trưng của toàn bộ tập dữ liệu (N, D).
            is_labeled: Mảng boolean (N,) đánh dấu các mẫu đã có nhãn.
            batch_size: Số lượng mẫu cần lấy thêm.
        """
        num_labeled = is_labeled.sum()
        target_budget = num_labeled + batch_size
        
        # 1. Tính độ điển hình (Typicality) cho toàn bộ không gian
        typicality = self.get_typicality(all_features)
        
        # 2. Phân cụm toàn bộ dữ liệu thành K cụm (K = Tổng ngân sách mục tiêu)
        print(f"TypiClust: Phân toàn bộ dữ liệu thành {target_budget} cụm...")
        kmeans = KMeans(n_clusters=target_budget, random_state=42, n_init="auto")
        cluster_ids = kmeans.fit_predict(all_features)
        
        # 3. Tính toán kích thước của từng cụm
        cluster_sizes = np.bincount(cluster_ids, minlength=target_budget)
        
        # Đếm số lượng mẫu đã có nhãn trong từng cụm
        labeled_indices = np.where(is_labeled)[0]
        labeled_clusters = cluster_ids[labeled_indices] if len(labeled_indices) > 0 else []
        labeled_counts_per_cluster = np.bincount(labeled_clusters, minlength=target_budget)
        
        # 4. Sắp xếp các cụm theo kích thước giảm dần (Ưu tiên cụm to nhất, dày đặc nhất)
        sorted_clusters = np.argsort(cluster_sizes)[::-1]
        
        selected_indices = []
        
        # 5. Chọn ra đúng `batch_size` mẫu
        for cluster_idx in sorted_clusters:
            # Nguyên tắc cốt lõi: CHỈ chọn mẫu từ cụm CHƯA CÓ bất kỳ nhãn nào
            if labeled_counts_per_cluster[cluster_idx] == 0:
                # Tìm các mẫu chưa có nhãn thuộc cụm này
                in_cluster_mask = (cluster_ids == cluster_idx) & (~is_labeled)
                in_cluster_indices = np.where(in_cluster_mask)[0]
                
                if len(in_cluster_indices) > 0:
                    # Chọn 1 mẫu có TÍNH ĐIỂN HÌNH (Typicality) CAO NHẤT trong cụm đó
                    cluster_typicality = typicality[in_cluster_indices]
                    best_local_idx = np.argmax(cluster_typicality)
                    best_global_idx = in_cluster_indices[best_local_idx]
                    
                    selected_indices.append(best_global_idx)
                    
                    if len(selected_indices) == batch_size:
                        break
                        
        # Fallback (Phòng hờ trường hợp K-Means bị lỗi phân phối khiến không đủ cụm rỗng)
        if len(selected_indices) < batch_size:
            remaining_unlabeled = np.where(~is_labeled)[0]
            remaining_unlabeled = np.setdiff1d(remaining_unlabeled, selected_indices)
            missing = batch_size - len(selected_indices)
            fallback_indices = np.random.choice(remaining_unlabeled, missing, replace=False)
            selected_indices.extend(fallback_indices)
            
        return np.array(selected_indices)

In [11]:
import torch
import numpy as np
from tqdm import tqdm

class CoreSetSelector:
    def __init__(self, budget, device="cuda"):
        """
        Khởi tạo thuật toán CoreSet (k-Center-Greedy).
        
        Args:
            budget (int): Số lượng mẫu cần lấy thêm để gán nhãn.
            device (str): Nơi tính toán ('cuda' hoặc 'cpu').
        """
        self.budget = budget
        self.device = device

    def select(self, unlabeled_features, labeled_features=None):
        """
        Chọn các mẫu chưa nhãn sao cho khoảng cách từ chúng tới tập đã có nhãn là xa nhất.
        
        Args:
            unlabeled_features (np.array): Đặc trưng của tập chưa gán nhãn (N, D).
            labeled_features (np.array): Đặc trưng của tập đã gán nhãn (M, D).
        """
        # Chuyển dữ liệu sang Tensor để tính toán song song trên GPU cho nhanh
        unlabeled_tensor = torch.tensor(unlabeled_features, device=self.device, dtype=torch.float32)
        
        selected_indices = []
        
        # 1. Khởi tạo khoảng cách tối thiểu từ mỗi điểm chưa nhãn tới tập đã nhãn
        if labeled_features is None or len(labeled_features) == 0:
            # Nếu chưa có bất kỳ nhãn nào, chọn ngẫu nhiên điểm đầu tiên làm tâm
            first_idx = np.random.randint(0, len(unlabeled_features))
            selected_indices.append(first_idx)
            
            # Tính khoảng cách từ các điểm còn lại tới điểm đầu tiên này
            center_feature = unlabeled_tensor[first_idx].unsqueeze(0)
            min_distances = torch.cdist(unlabeled_tensor, center_feature).squeeze()
            
            # Đã chọn 1 điểm, nên vòng lặp sẽ trừ đi 1
            loop_budget = self.budget - 1
        else:
            labeled_tensor = torch.tensor(labeled_features, device=self.device, dtype=torch.float32)
            
            # Tính ma trận khoảng cách giữa tập chưa nhãn và tập đã nhãn
            # dist_matrix shape: (Số mẫu unlabel, Số mẫu label)
            dist_matrix = torch.cdist(unlabeled_tensor, labeled_tensor)
            
            # Lấy khoảng cách nhỏ nhất từ mỗi điểm unlabel tới tập label
            min_distances, _ = torch.min(dist_matrix, dim=1)
            loop_budget = self.budget

        print(f"Đang chạy thuật toán CoreSet (k-Center-Greedy) để chọn {self.budget} mẫu...")
        
        # 2. Vòng lặp k-Center-Greedy
        for _ in tqdm(range(loop_budget)):
            # Chọn điểm có "khoảng cách tối thiểu" lớn nhất (nằm xa tập label nhất)
            furthest_idx = torch.argmax(min_distances).item()
            selected_indices.append(furthest_idx)
            
            # Lấy đặc trưng của điểm vừa chọn
            new_center = unlabeled_tensor[furthest_idx].unsqueeze(0)
            
            # Tính khoảng cách từ điểm vừa chọn tới toàn bộ tập unlabel
            new_distances = torch.cdist(unlabeled_tensor, new_center).squeeze()
            
            # Cập nhật lại mảng min_distances
            min_distances = torch.minimum(min_distances, new_distances)

        return np.array(selected_indices)

In [12]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

def run_pipeline_coreset(
    train_embeddings, 
    train_labels, 
    init_budget=100, 
    budget_per_iter=50, 
    max_iterations=4,
    device="cuda"
):
    """
    Quy trình Active Learning sử dụng thuật toán CoreSet.
    """
    num_samples = len(train_embeddings)
    is_labeled = np.zeros(num_samples, dtype=bool)
    queried_labels = np.full(num_samples, -1)
    
    # Tiền xử lý PCA để giảm nhiễu khoảng cách L2 trong không gian đa chiều cao
    pca = PCA(n_components=128, random_state=42)
    train_features_pca = pca.fit_transform(train_embeddings)
    
    selector = CoreSetSelector(budget=budget_per_iter, device=device)
    
    # --- BƯỚC 1: COLD START ---
    # CoreSet có thể tự chọn mẫu đầu tiên nếu truyền labeled_features = None
    print(f"=== BƯỚC 1: Khởi tạo {init_budget} nhãn bằng CoreSet ===")
    selector.budget = init_budget
    initial_indices = selector.select(train_features_pca, labeled_features=None)
    
    is_labeled[initial_indices] = True
    queried_labels[initial_indices] = train_labels[initial_indices]
    
    # Đặt lại budget cho các vòng lặp sau
    selector.budget = budget_per_iter
    
    # --- BƯỚC 2: VÒNG LẶP ACTIVE LEARNING ---
    for iteration in range(max_iterations):
        current_count = np.sum(is_labeled)
        print(f"\n=== Vòng CoreSet {iteration + 1} | Tổng nhãn: {current_count} ===")
        
        labeled_idx = np.where(is_labeled)[0]
        unlabeled_idx = np.where(~is_labeled)[0]
        
        if len(unlabeled_idx) == 0: break
            
        labeled_features = train_features_pca[labeled_idx]
        unlabeled_features = train_features_pca[unlabeled_idx]
        
        # Chạy thuật toán k-Center-Greedy
        new_local_indices = selector.select(
            unlabeled_features=unlabeled_features, 
            labeled_features=labeled_features
        )
        
        new_global_indices = unlabeled_idx[new_local_indices]
        
        # Gán nhãn cho các mẫu được chọn
        is_labeled[new_global_indices] = True
        queried_labels[new_global_indices] = train_labels[new_global_indices]
        
    # --- BƯỚC 3: HUẤN LUYỆN PROXY MODEL CUỐI CÙNG ---
    print(f"\nHuấn luyện mô hình cuối cùng trên {np.sum(is_labeled)} nhãn...")
    final_labeled_idx = np.where(is_labeled)[0]
    X_train_final = train_features_pca[final_labeled_idx]
    y_train_final = queried_labels[final_labeled_idx]
    
    final_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    final_model.fit(X_train_final, y_train_final)
    
    return final_model, pca

In [13]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

def run_pipeline_typiclust(
    train_embeddings, 
    train_labels, 
    init_budget=100, 
    budget_per_iter=50, 
    max_iterations=4
):
    """
    Quy trình Active Learning toàn diện sử dụng TypiClust.
    """
    num_samples = len(train_embeddings)
    
    # Tiền xử lý PCA để khoảng cách Euclidean (trong k-NN và k-Means) chính xác hơn
    pca = PCA(n_components=128, random_state=42)
    train_features_pca = pca.fit_transform(train_embeddings)
    
    is_labeled = np.zeros(num_samples, dtype=bool)
    queried_labels = np.full(num_samples, -1)
    
    # Khởi tạo thuật toán
    selector = TypiClustSelector(k_nn=20)
    
    # --- BƯỚC 1: COLD START BẰNG TYPICLUST ---
    print(f"=== BƯỚC 1: Khởi tạo {init_budget} nhãn bằng TypiClust (Cold Start) ===")
    # Chỉ việc truyền mảng is_labeled rỗng, TypiClust sẽ chọn ra các cụm to nhất
    initial_indices = selector.select(train_features_pca, is_labeled, batch_size=init_budget)
    
    is_labeled[initial_indices] = True
    queried_labels[initial_indices] = train_labels[initial_indices]
    
    # --- BƯỚC 2: VÒNG LẶP ACTIVE LEARNING ---
    for iteration in range(max_iterations):
        current_count = np.sum(is_labeled)
        print(f"\n=== Vòng TypiClust {iteration + 1} | Tổng nhãn: {current_count} ===")
        
        if np.all(is_labeled): break
            
        # TypiClust cần quan sát cả tập đã có nhãn và chưa nhãn để chia cụm lại
        new_indices = selector.select(train_features_pca, is_labeled, batch_size=budget_per_iter)
        
        is_labeled[new_indices] = True
        queried_labels[new_indices] = train_labels[new_indices]
        print(f" + Đã query thêm {len(new_indices)} ảnh đại diện cao (Typical).")
        
    # --- BƯỚC 3: HUẤN LUYỆN PROXY MODEL CUỐI CÙNG ---
    print(f"\nHuấn luyện mô hình cuối cùng trên {np.sum(is_labeled)} nhãn...")
    final_labeled_idx = np.where(is_labeled)[0]
    X_train_final = train_features_pca[final_labeled_idx]
    y_train_final = queried_labels[final_labeled_idx]
    
    final_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    final_model.fit(X_train_final, y_train_final)
    
    return final_model, pca

In [14]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

def train_pipeline_alpha_mix(
    train_embeddings, 
    train_labels, 
    num_classes, 
    init_budget=100, 
    budget_per_iter=50, 
    max_iterations=4, 
    device="cuda"
):
    """
    Chỉ thực hiện huấn luyện Active Learning và trả về mô hình cuối cùng.
    """
    num_samples = len(train_embeddings)
    
    # --- BƯỚC 0: TIỀN XỬ LÝ PCA ---
    # Giảm chiều để ALFA-Mix hoạt động hiệu quả và tránh nhiễu
    pca = PCA(n_components=128, random_state=42)
    train_features_pca = pca.fit_transform(train_embeddings)
    
    is_labeled = np.zeros(num_samples, dtype=bool)
    queried_labels = np.full(num_samples, -1)
    
    # --- BƯỚC 1: COLD START (K-MEANS) ---
    print(f"--- BƯỚC 1: Khởi tạo {init_budget} nhãn bằng K-Means ---")
    kmeans = KMeans(n_clusters=init_budget, random_state=42, n_init="auto")
    distances = kmeans.fit_transform(train_features_pca)
    initial_indices = np.argmin(distances, axis=0)
    
    is_labeled[initial_indices] = True
    queried_labels[initial_indices] = train_labels[initial_indices]
    
    # Khởi tạo ALFA-Mix Selector
    # ALFA-Mix tìm kiếm các mẫu không nhãn có đặc trưng đủ khác biệt
    selector = ALFAMixSelector(budget=budget_per_iter, device=device)
    
    # --- BƯỚC 2: VÒNG LẶP ACTIVE LEARNING ---
    for iteration in range(max_iterations):
        current_count = np.sum(is_labeled)
        print(f"=== Vòng ALFA-Mix {iteration + 1} | Nhãn: {current_count} ===")
        
        labeled_idx = np.where(is_labeled)[0]
        unlabeled_idx = np.where(~is_labeled)[0]
        
        if len(unlabeled_idx) == 0: break
            
        # Chuẩn bị dữ liệu dạng Tensor cho Selector
        labeled_f = torch.from_numpy(train_features_pca[labeled_idx]).float()
        labeled_l = torch.from_numpy(queried_labels[labeled_idx]).long()
        unlabeled_f = torch.from_numpy(train_features_pca[unlabeled_idx]).float()
        
        # Chọn mẫu bằng cách tìm sự không nhất quán trong dự đoán khi trộn đặc trưng
        new_local_indices = selector.select(
            unlabeled_features=unlabeled_f,
            labeled_features=labeled_f,
            labeled_labels=labeled_l,
            num_classes=num_classes
        )
        
        new_global_indices = unlabeled_idx[new_local_indices]
        is_labeled[new_global_indices] = True
        queried_labels[new_global_indices] = train_labels[new_global_indices]
        
    # --- BƯỚC 3: HUẤN LUYỆN MÔ HÌNH CUỐI CÙNG ---
    print(f"Huấn luyện mô hình cuối cùng trên {np.sum(is_labeled)} nhãn...")
    final_labeled_idx = np.where(is_labeled)[0]
    X_train_final = train_features_pca[final_labeled_idx]
    y_train_final = queried_labels[final_labeled_idx]
    
    final_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    final_model.fit(X_train_final, y_train_final)
    
    return final_model, pca

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, pca, test_embeddings, test_labels):
    """
    Đánh giá mô hình đã huấn luyện trên tập test riêng biệt.
    """
    print("\n" + "="*40)
    print("BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST")
    
    # 1. Chuyển đổi tập test sang không gian PCA đã học
    X_test_pca = pca.transform(test_embeddings)
    
    # 2. Dự đoán
    y_pred = model.predict(X_test_pca)
    
    # 3. Tính toán các chỉ số
    acc = accuracy_score(test_labels, y_pred)
    pre = precision_score(test_labels, y_pred, average="macro", zero_division=0)
    rec = recall_score(test_labels, y_pred, average="macro", zero_division=0)
    f1 = f1_score(test_labels, y_pred, average="macro", zero_division=0)
    
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {pre * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"Macro F1  : {f1 * 100:.2f}%")
    print("="*40 + "\n")
    
    return {"accuracy": acc, "f1": f1}

In [16]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
def evaluate_active_ft_results(queried_features, queried_labels, test_embeddings, test_labels):
    """
    Đánh giá hiệu quả của tập mẫu được chọn bởi ActiveFT trên tập Test với đầy đủ chỉ số.
    """
    # 1. Khởi tạo PCA và Proxy Model
    # Sử dụng n_components=128 để giảm nhiễu nhưng vẫn giữ đủ đặc trưng tuyến tính [cite: 9]
    pca = PCA(n_components=128, random_state=42)
    proxy_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)

    # 2. Xử lý dữ liệu qua PCA
    X_train_pca = pca.fit_transform(queried_features)
    X_test_pca = pca.transform(test_embeddings)

    # 3. Huấn luyện Proxy Model
    # Phương pháp Single-pass của ActiveFT giúp huấn luyện cực nhanh so với AL truyền thống [cite: 40, 334]
    print(f"--- Đang huấn luyện Proxy Model trên {len(queried_labels)} mẫu ActiveFT ---")
    proxy_model.fit(X_train_pca, queried_labels)

    # 4. Dự đoán trên tập Test biệt lập
    preds = proxy_model.predict(X_test_pca)
    
    # 5. Tính toán các chỉ số (Sử dụng average='macro' để đánh giá công bằng giữa các lớp)
    acc = accuracy_score(test_labels, preds)
    pre = precision_score(test_labels, preds, average="macro", zero_division=0)
    rec = recall_score(test_labels, preds, average="macro", zero_division=0)
    f1 = f1_score(test_labels, preds, average="macro", zero_division=0)

    # Hiển thị kết quả
    print("\n" + "="*35)
    print(f"KẾT QUẢ ĐÁNH GIÁ (TEST SET)")
    print("-" * 35)
    print(f"Accuracy  : {acc * 100:.2f}%")
    print(f"Precision : {pre * 100:.2f}%")
    print(f"Recall    : {rec * 100:.2f}%")
    print(f"Macro F1  : {f1 * 100:.2f}%")
    print("="*35 + "\n")
    
    return proxy_model, pca

In [17]:
print(O)

NameError: name 'O' is not defined

In [18]:
train_mnist_loader, test_mnist_loader, class_mnist_names = get_data_loaders(PATHMNIST_PATH)

train_mnist_embeddings, train_mnist_labels = extract_embeddings(train_mnist_loader)
test_mnist_embeddings, test_mnist_labels = extract_embeddings(test_mnist_loader)
trained_model, trained_pca = run_pipeline_coreset(
    train_mnist_embeddings, 
    train_mnist_labels, 
    init_budget=100,       
    budget_per_iter=25, 
    max_iterations=4
)

evaluate_model(
    trained_model, 
    trained_pca, 
    test_mnist_embeddings, 
    test_mnist_labels
)
del train_mnist_loader, test_mnist_loader, class_mnist_names
del train_mnist_embeddings, train_mnist_labels
del test_mnist_embeddings, test_mnist_labels
clear_memory()

Train size: 100000 | Test size: 7180
Loading PLIP model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: vinid/plip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_ids                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        

Loading PLIP model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...


=== Vòng CoreSet 3 | Tổng nhãn: 150 ===
Đang chạy thuật toán CoreSet (k-Center-Greedy) để chọn 25 mẫu...


100%|██████████| 25/25 [00:00<00:00, 440.99it/s]



=== Vòng CoreSet 4 | Tổng nhãn: 175 ===
Đang chạy thuật toán CoreSet (k-Center-Greedy) để chọn 25 mẫu...


100%|██████████| 25/25 [00:00<00:00, 434.86it/s]



Huấn luyện mô hình cuối cùng trên 200 nhãn...

BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST
Accuracy  : 90.03%
Precision : 87.82%
Recall    : 87.01%
Macro F1  : 86.80%



In [19]:
train_histo_loader, test_histo_loader, class_histo_names = get_data_loaders(HISTOSET_PATH)

train_histo_embeddings, train_histo_labels = extract_embeddings(train_histo_loader)
test_histo_embeddings, test_histo_labels = extract_embeddings(test_histo_loader)

trained_model, trained_pca = run_pipeline_coreset(
    train_histo_embeddings, 
    train_histo_labels, 
    init_budget=100,       
    budget_per_iter=50, 
    max_iterations=4
)
evaluate_model(
    trained_model, 
    trained_pca, 
    test_histo_embeddings, 
    test_histo_labels
)

del train_histo_loader, test_histo_loader, class_histo_names
del train_histo_embeddings, train_histo_labels
del test_histo_embeddings, test_histo_labels
clear_memory()

Train size: 22400 | Test size: 5600
Loading PLIP model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

CLIPVisionModelWithProjection LOAD REPORT from: vinid/plip
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.embeddings.position_ids                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        

Loading PLIP model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

...


=== Vòng CoreSet 3 | Tổng nhãn: 200 ===
Đang chạy thuật toán CoreSet (k-Center-Greedy) để chọn 50 mẫu...


100%|██████████| 50/50 [00:00<00:00, 1819.61it/s]



=== Vòng CoreSet 4 | Tổng nhãn: 250 ===
Đang chạy thuật toán CoreSet (k-Center-Greedy) để chọn 50 mẫu...


100%|██████████| 50/50 [00:00<00:00, 1846.17it/s]



Huấn luyện mô hình cuối cùng trên 300 nhãn...

BẮT ĐẦU ĐÁNH GIÁ TRÊN TẬP TEST
Accuracy  : 84.61%
Precision : 84.96%
Recall    : 84.44%
Macro F1  : 84.43%



In [ ]:
train_mnist_loader, test_mnist_loader, class_mnist_names = get_data_loaders(PATHMNIST_PATH)

train_mnist_embeddings, train_mnist_labels = extract_embeddings(train_mnist_loader)
test_mnist_embeddings, test_mnist_labels = extract_embeddings(test_mnist_loader)
trained_model, trained_pca = run_pipeline_typiclust(
    train_mnist_embeddings, 
    train_mnist_labels, 
    init_budget=100,       
    budget_per_iter=25, 
    max_iterations=4
)

evaluate_model(
    trained_model, 
    trained_pca, 
    test_mnist_embeddings, 
    test_mnist_labels
)
del train_mnist_loader, test_mnist_loader, class_mnist_names
del train_mnist_embeddings, train_mnist_labels
del test_mnist_embeddings, test_mnist_labels
clear_memory()

In [ ]:
train_histo_loader, test_histo_loader, class_histo_names = get_data_loaders(HISTOSET_PATH)

train_histo_embeddings, train_histo_labels = extract_embeddings(train_histo_loader)
test_histo_embeddings, test_histo_labels = extract_embeddings(test_histo_loader)

trained_model, trained_pca = run_pipeline_typiclust(
    train_histo_embeddings, 
    train_histo_labels, 
    init_budget=100,       
    budget_per_iter=50, 
    max_iterations=4
)
evaluate_model(
    trained_model, 
    trained_pca, 
    test_histo_embeddings, 
    test_histo_labels
)

del train_histo_loader, test_histo_loader, class_histo_names
del train_histo_embeddings, train_histo_labels
del test_histo_embeddings, test_histo_labels
clear_memory()

In [ ]:
train_mnist_loader, test_mnist_loader, class_mnist_names = get_data_loaders(PATHMNIST_PATH)

train_mnist_embeddings, train_mnist_labels = extract_embeddings(train_mnist_loader)
test_mnist_embeddings, test_mnist_labels = extract_embeddings(test_mnist_loader)

trained_model, trained_pca = train_pipeline_alpha_mix(
    train_mnist_embeddings, 
    train_mnist_labels, 
    num_classes=9,
    init_budget=100, 
    budget_per_iter=25, 
    max_iterations=4,
    device=DEVICE
)
results = evaluate_model(
    trained_model, 
    trained_pca, 
    test_mnist_embeddings, 
    test_mnist_labels
)

del train_mnist_loader, test_mnist_loader, class_mnist_names
del train_mnist_embeddings, train_mnist_labels
del test_mnist_embeddings, test_mnist_labels
clear_memory()

In [ ]:
train_histo_loader, test_histo_loader, class_histo_names = get_data_loaders(HISTOSET_PATH)

train_histo_embeddings, train_histo_labels = extract_embeddings(train_histo_loader)
test_histo_embeddings, test_histo_labels = extract_embeddings(test_histo_loader)

trained_model, trained_pca = train_pipeline_alpha_mix(
    train_histo_embeddings, 
    train_histo_labels, 
    num_classes=14,
    init_budget=100, 
    budget_per_iter=50, 
    max_iterations=4,
    device=DEVICE
)
results = evaluate_model(
    trained_model, 
    trained_pca, 
    test_histo_embeddings, 
    test_histo_labels
)


del train_histo_loader, test_histo_loader, class_histo_names
del train_histo_embeddings, train_histo_labels
del test_histo_embeddings, test_histo_labels
clear_memory()

In [ ]:
print(O)

In [ ]:
train_mnist_loader, test_mnist_loader, class_mnist_names = get_data_loaders(PATHMNIST_PATH)

train_mnist_embeddings, train_mnist_labels = extract_embeddings(train_mnist_loader)
test_mnist_embeddings, test_mnist_labels = extract_embeddings(test_mnist_loader)

features_tensor = torch.from_numpy(train_mnist_embeddings).to(DEVICE).float()
selector = ActiveFTSelector(budget=200, iterations=200, lr=1e-2)
selected_indices = selector.select(features_tensor)

train_labels_queried = train_mnist_labels[selected_indices]
train_features_queried = train_mnist_embeddings[selected_indices]

trained_model, trained_pca = evaluate_active_ft_results(
    train_features_queried, 
    train_labels_queried, 
    test_mnist_embeddings, 
    test_mnist_labels
)


del train_mnist_loader, test_mnist_loader, class_mnist_names
del train_mnist_embeddings, train_mnist_labels
del test_mnist_embeddings, test_mnist_labels
clear_memory()

In [ ]:
train_histo_loader, test_histo_loader, class_histo_names = get_data_loaders(HISTOSET_PATH)

train_histo_embeddings, train_histo_labels = extract_embeddings(train_histo_loader)
test_histo_embeddings, test_histo_labels = extract_embeddings(test_histo_loader)

features_tensor = torch.from_numpy(train_histo_embeddings).to(DEVICE).float()
selector = ActiveFTSelector(budget=300, iterations=200, lr=1e-2)
selected_indices = selector.select(features_tensor)

train_labels_queried = train_histo_labels[selected_indices]
train_features_queried = train_histo_embeddings[selected_indices]

trained_model, trained_pca = evaluate_active_ft_results(
    train_features_queried, 
    train_labels_queried, 
    test_histo_embeddings, 
    test_histo_labels
)


del train_histo_loader, test_histo_loader, class_histo_names
del train_histo_embeddings, train_histo_labels
del test_histo_embeddings, test_histo_labels
clear_memory()